__Log Likelihood Function: Pixelization__

This script provides a step-by-step guide of the **PyAutoLens** `log_likelihood_function` which is used to fit
`Imaging` data with a pixelization (`RectangularUniform` mesh and `Constant` regularization scheme`).

This script has the following aims:

 - To provide a resource that authors can include in papers using **PyAutoLens**, so that readers can understand the
 likelihood function (including references to the previous literature from which it is defined) without having to
 write large quantities of text and equations.

 - To make pixelized reconstructions less of a "black-box" to users.

__Contents__

- **Simplifications:** This example uses a `RectangularUniform` mesh, where all rectangular source pixels have the same.
- **Prerequisites:** The likelihood function of a pixelization builds on that used for standard light profiles and.
- **Dataset & Mask:** Standard set up of the dataset and mask that is fitted.
- **Over Sampling:** Set up the adaptive over-sampling grid for accurate light profile evaluation.
- **Masked Image Grid:** To perform lensing calculations we first must define the 2D image-plane (y,x) coordinates used in.
- **Mesh Shape:** The `mesh_shape` parameter defines number of pixels used by the rectangular mesh to reconstruct the.
- **Edge Zeroing:** By default, all pixels at the edge of the mesh in the source-plane are forced to solutions of zero.
- **Lens Galaxy:** We set up a lens galaxy with the lens light and mass, which we will use to demonstrate a pixelized.
- **Source Galaxy Pixelization and Regularization:** The source galaxy is reconstructed using a pixel-grid, in this example a RectangularUniform mesh.
- **Lens Light:** Compute a 2D image of the lens galaxy's light as the sum of its individual light profiles (the.
- **Ray Tracing:** To perform lensing calculations we ray-trace every 2d (y,x) coordinate $\theta$ from the.
- **Border Relocation:** Coordinates that are ray-traced near the mass profile centres are heavily demagnified and may trace.
- **Source Pixel Centre Calculation:** In order to reconstruct the source galaxy using a RectangularUniform mesh, we need to determine the.
- **Interpolation:** We now combine grids computed above to create an `Interpolator`, which describes how image grid.
- **Mapper:** We now use the interpolator to create a `Mapper`, which describes the mapping between every image.
- **Alternative Meshes:** We can briefly consider how this step differs for other mesh types.
- **Mapping Matrix:** The `mapping_matrix` represents the image-pixel to source-pixel mappings above in a 2D matrix.
- **Image Reconstruction:** Using the reconstructed source pixel fluxes we can map the source reconstruction back to the image.
- **Likelihood Function:** We now quantify the goodness-of-fit of our lens model and source reconstruction.
- **Chi Squared:** The first term is a $\chi^2$ statistic, which is defined above in our merit function as and is.
- **Regularization Term:** The second term, $s^{T} H s$, corresponds to the $\lambda $G_{\rm L}$ regularization term we added.
- **Complexity Terms:** Up to this point, it is unclear why we chose a value of `regularization_coefficient=1.0`.
- **Noise Normalization Term:** Our likelihood function assumes the imaging data consists of independent Gaussian noise in every.
- **Calculate The Log Likelihood:** We can now, finally, compute the `log_likelihood` of the lens model, by combining the five terms.
- **Fit:** Fit the lens model to the dataset.
- **Lens Modeling:** To fit a lens model to data, the likelihood function illustrated in this tutorial is sampled using.
- **Wrap Up:** Summary of the script and next steps.

__Simplifications__

This example uses a `RectangularUniform` mesh, where all rectangular source pixels have the same size. Most
pixelization examples use a `RectangularBilinearAdaptDensity` mesh, which adapts the size of source pixels to the
density of points in the source-plane (e.g. the caustic).

The `RectangularUniform` mesh is used here because it is simpler to explain the likelihood function
and illustrate the key steps in the calculation. The same principles apply to other mesh types, which this
example will explain where relevant.

__Prerequisites__

The likelihood function of a pixelization builds on that used for standard light profiles and
linear light profiles, therefore you must read the following notebooks before this script:

- `imaging/likelihood_function.ipynb`.
- `imaging/features/linear_light_profiles/likelihood_function.ipynb`.

__Google Colab Setup__

This cell sets up the environment when the notebook is run on Google Colab: it installs the
required PyAuto packages, clones the workspace (configuration files and example datasets) and
points the configuration at it. If you are running the notebook elsewhere (e.g. locally via
your own installation) it does nothing, and you can run it safely.

Colab tip: model-fits run much faster on a GPU — enable one via "Runtime" -> "Change runtime
type" -> "Hardware accelerator" before running the notebook.

In [ ]:
try:
    import google.colab
except ImportError:
    from autolens import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autolens")

In [ ]:

from autolens import jax_wrapper  # Sets JAX environment before other imports

from autolens import setup_notebook; setup_notebook()

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

import autolens as al
import autoarray.plot as aaplt
import autolens.plot as aplt

__Dataset__

In order to perform a likelihood evaluation, we first load a dataset.

This example fits a simulated strong lens which is simulated using a 0.1 arcsecond-per-pixel resolution (this is lower
resolution than the best quality Hubble Space Telescope imaging and close to that of the Euclid space satellite).

In [ ]:
dataset_path = Path("dataset", "imaging", "simple")

__Dataset Auto-Simulation__

If the dataset does not already exist on your system, it will be created by running the corresponding
simulator script. This ensures that all example scripts can be run without manually simulating data first.

In [ ]:
if al.util.dataset.should_simulate(str(dataset_path)):
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "scripts/imaging/simulator.py"],
        check=True,
    )

dataset = al.Imaging.from_fits(
    data_path=dataset_path / "data.fits",
    psf_path=dataset_path / "psf.fits",
    noise_map_path=dataset_path / "noise_map.fits",
    pixel_scales=0.1,
)

I will use in-built visualization tools for plotting. 

For example, using the `aplt.subplot_imaging_dataset` I can plot the imaging dataset we performed a likelihood evaluation on.

In [ ]:
aplt.subplot_imaging_dataset(dataset=dataset)

__Mask__

The likelihood is only evaluated using image pixels contained within a 2D mask, which we choose before performing
lens modeling.

Below, we define a 2D circular mask with a 3.0" radius.

In [ ]:
mask_radius = 3.0

mask = al.Mask2D.circular(
    shape_native=dataset.shape_native,
    pixel_scales=dataset.pixel_scales,
    radius=mask_radius,
)

masked_dataset = dataset.apply_mask(mask=mask)

When we plot the masked imaging, only the circular masked region is shown.

In [ ]:
aplt.subplot_imaging_dataset(dataset=masked_dataset)

__Over Sampling__

Over sampling evaluates a light profile using multiple samples of its intensity per image-pixel.

For simplicity, we disable over sampling in this guide by setting `sub_size=1`. 

a full description of over sampling and how to use it is given in `autolens_workspace/*/guides/advanced/over_sampling.py`.

This example will explain how sub-gridding changes the linear algebra at the relevent section.

In [ ]:
masked_dataset = masked_dataset.apply_over_sampling(
    over_sample_size_lp=1,
    over_sample_size_pixelization=1,
)

__Masked Image Grid__

To perform lensing calculations we first must define the 2D image-plane (y,x) coordinates used in the calculation.

For light profiles these are given by `masked_dataset.lp`, which is a uniform grid of (y,x) Cartesian coordinates
which have had the 3.0" circular mask applied.

A pixelization uses a separate grid of (y,x) coordinates, called `masked_dataset.grids.pixelization`, which is
identical to the light profile grid but may of had a different over-sampling scale applied (but in this example
does not).

Each (y,x) coordinate coordinates to the centre of each image-pixel in the dataset, meaning that when this grid is
used to construct a pixelization there is a straight forward mapping between the image data and pixelization pixels.

In [ ]:
aplt.plot_grid(grid=masked_dataset.grids.pixelization, title="")

__Mesh Shape__

The `mesh_shape` parameter defines number of pixels used by the rectangular mesh to reconstruct the source,
set below to 28 x 28. 

The `mesh_shape` must be fixed before modeling and cannot be a free parameter of the model, because JAX uses the
mesh shape to define static shaped arrays which use the mesh to reconstruct the source. For a rectangular
mesh, the same number of pixels must be used in the y and x directions.

__Edge Zeroing__

By default, all pixels at the edge of the mesh in the source-plane are forced to solutions of zero brightness by 
the linear algebra solver. This prevents unphysical solutions where pixels at the edge of the mesh reconstruct 
bright surface brightnesses, often because they fit residuals from the lens light subtraction.

For a rectangular mesh, the source code computes edge pixels internally using the known
pixels at the edge of the mesh. 

In [ ]:
mesh_pixels_yx = 28
mesh_shape = (mesh_pixels_yx, mesh_pixels_yx)

__Lens Galaxy__

We set up a lens galaxy with the lens light and mass, which we will use to demonstrate a pixelized source
reconstruction.

In [ ]:
bulge = al.lp.Sersic(
    centre=(0.0, 0.0),
    ell_comps=al.convert.ell_comps_from(axis_ratio=0.9, angle=45.0),
    intensity=2.0,
    effective_radius=0.6,
    sersic_index=3.0,
)

mass = al.mp.Isothermal(
    centre=(0.0, 0.0),
    einstein_radius=1.6,
    ell_comps=al.convert.ell_comps_from(axis_ratio=0.9, angle=45.0),
)

shear = al.mp.ExternalShear(gamma_1=0.05, gamma_2=0.05)

lens_galaxy = al.Galaxy(redshift=0.5, bulge=bulge, mass=mass)

# The external shear is a property of the system, not of a galaxy, so it is held in a
# `MassField` given to the `Tracer` via `fields=` (see `imaging/modeling.py`).

field = al.MassField(redshift=0.5, shear=shear)

__Source Galaxy Pixelization and Regularization__

The source galaxy is reconstructed using a pixel-grid, in this example a RectangularUniform mesh, which accounts for 
irregularities and asymmetries in the source's surface brightness. 

A constant regularization scheme is applied which applies a smoothness prior on the reconstruction. 

In [ ]:
pixelization = al.Pixelization(
    mesh=al.mesh.RectangularUniform(shape=mesh_shape),
    regularization=al.reg.Constant(coefficient=1.0),
)

source_galaxy = al.Galaxy(redshift=1.0, pixelization=pixelization)

__Lens Light__

Compute a 2D image of the lens galaxy's light as the sum of its individual light profiles (the `Sersic` 
bulge). 

This computes the `image` of each `LightProfile` and adds them together. 

In [ ]:
image = lens_galaxy.image_2d_from(grid=masked_dataset.grid)


To convolve the lens's 2D image with the imaging data's PSF, we need its `blurring_image`. This represents all flux 
values not within the mask, which are close enough to it that their flux blurs into the mask after PSF convolution.

To compute this, a `blurring_mask` and `blurring_grid` are used, corresponding to these pixels near the edge of the 
actual mask whose light blurs into the image:

In [ ]:
blurring_image_2d = lens_galaxy.image_2d_from(grid=masked_dataset.grids.blurring)


__Lens Light Convolution + Subtraction__

Convolve the 2D lens light images above with the PSF using a `Convolver`.

In [ ]:
convolved_image_2d = masked_dataset.psf.convolved_image_from(
    image=image, blurring_image=blurring_image_2d
)

aplt.plot_array(array=convolved_image_2d, title="")

We can now subtract this image from the observed image to produce a `lens_subtracted_image`:

In [ ]:
lens_subtracted_image = masked_dataset.data - convolved_image_2d

aplt.plot_array(array=lens_subtracted_image, title="")

__Ray Tracing__

To perform lensing calculations we ray-trace every 2d (y,x) coordinate $\theta$ from the image-plane to its (y,x) 
source-plane coordinate $\beta$ using the summed deflection angles $\alpha$ of the mass profiles:

 $\beta = \theta - \alpha(\theta)$

The likelihood function of a pixelized source reconstruction ray-traces one grid from the image-plane to the source-plane:

 1) A 2D grid of (y,x) coordinates aligned with the imaging data's image-pixels.
 
The function below computes the 2D deflection angles of the tracer's lens galaxies and subtracts them from the 
image-plane 2D (y,x) coordinates $\theta$ of each grid, thus ray-tracing their coordinates to the source plane to 
compute their $\beta$ values.

In [ ]:
tracer = al.Tracer(galaxies=[lens_galaxy, source_galaxy], fields=[field])

The source code gets quite complex when handling grids for a pixelization, but it is all handled in
the `TracerToInversion` objects.

The plots at the bottom of this cell show the traced grids used by the source pixelization, showing
how the traced image pixels are constructed.

In [ ]:
tracer_to_inversion = al.TracerToInversion(tracer=tracer, dataset=masked_dataset)

# A list of every grid (e.g. image-plane, source-plane) however we only need the source plane grid with index -1.
traced_grid_pixelization = tracer.traced_grid_2d_list_from(
    grid=masked_dataset.grids.pixelization
)[-1]


aplt.plot_grid(grid=traced_grid_pixelization, title="")

__Border Relocation__

Coordinates that are ray-traced near the mass profile centres are heavily demagnified and may trace to far outskirts of
the source-plane. 

We relocate these pixels (for both grids above) to the edge of the source-plane border (defined via the border of the 
image-plane mask). This is detailed in **HowToLens chapter 3 tutorial 6** and figure 2 of https://arxiv.org/abs/1708.07377.

In [ ]:
from autoarray.inversion.mesh.border_relocator import BorderRelocator

border_relocator = BorderRelocator(mask=masked_dataset.mask, sub_size=1)

relocated_grid = border_relocator.relocated_grid_from(grid=traced_grid_pixelization)


aplt.plot_grid(grid=relocated_grid, title="")

__Source Pixel Centre Calculation__

In order to reconstruct the source galaxy using a RectangularUniform mesh, we need to determine the centres of 
the rectangular mesh's source pixels.

We do this by overlying a rectangular grid on the relocated traced image-plane grid computed above.

This distributes the rectangular mesh so it fully overlaps the region of the source-plane containing the traced 
image-pixels without having edge pixels that extend beyond this region.

In [ ]:
from autoarray.inversion.mesh.mesh.rectangular_rtu_adapt_density import (
    overlay_grid_from,
)

mesh_grid = overlay_grid_from(
    shape_native=mesh_shape, grid=al.Grid2DIrregular(relocated_grid)
)

__Interpolation__

We now combine grids computed above to create an `Interpolator`, which describes how image grid pixel maps to
every rectangular mesh pixel. 

In [ ]:
interpolator = pixelization.mesh.interpolator_from(
    source_plane_data_grid=relocated_grid,
    source_plane_mesh_grid=mesh_grid,
)

For the rectangular mesh, the interpolation scheme is called bilinear interpolation, which means that every image 
pixel maps to the rectangular pixel it lands in and the three neighboring rectangular pixels. 

The weight of each mapping is determined by the bilinear interpolation scheme, which is a function of how close the 
image pixel is to the centre of the rectangular pixel it lands in and the three neighboring rectangular pixels.

We can print the mappings and weights, for example of the first image pixel, to confirm this.

In [ ]:
print(interpolator.mappings[0])
print(interpolator.weights[0])

__Mapper__

We now use the interpolator to create a `Mapper`, which describes the mapping between every image pixel and every 
rectangular pixel, based on the interpolation scheme above.

In [ ]:
mapper = al.Mapper(interpolator=interpolator)

Plotting the RectangularUniform mesh shows that the source-plane and been discretized into a grid of rectangular pixels.

Below, we plot the RectangularUniform mesh without the traced image-grid pixels (for clarity) and with them as black 
dots in order to show how each set of image-pixels fall within a RectangularUniform pixel.

In [ ]:
aplt.plot_grid(grid=mapper.source_plane_mesh_grid, title="Source-Plane Mesh Grid")

aplt.plot_grid(grid=mapper.source_plane_mesh_grid, title="Source-Plane Mesh Grid")


The `Mapper` contains:

 1) `source_plane_data_grid`: the traced grid of (y,x) image-pixel coordinate centres (`relocated_grid`).
 2) `source_plane_mesh_grid`: The RectangularUniform mesh of traced (y,x) source-pixel coordinates (`mesh_grid`).

We have therefore discretized the source-plane into a rectangular mesh, and can pair every traced image-pixel 
coordinate with the corresponding source pixel it lands in.

These quantities are both in the source-plane, and do not by themselves describe the mapping between the image and 
source planes. The mapping is described by the `pix_indexes_for_sub_slim_index`, which maps every image-pixel index to 
every pixelization pixel index.

`pix_indexes` refers to the pixelization pixel indexes (e.g. rectangular pixel 0, 1, 2 etc.) and `sub_slim_index`  
refers to the index of an image pixel (e.g. image-pixel 0, 1, 2 etc.). 

For example, printing the first ten entries of `pix_indexes_for_sub_slim_index` shows the first ten source-pixel
indexes these image sub-pixels map too.

In [ ]:
pix_indexes_for_sub_slim_index = mapper.pix_indexes_for_sub_slim_index

print(pix_indexes_for_sub_slim_index[0:9])

This array can be used to visualize how an input list of image-pixel indexes map to the source-plane.

It also shows that image-pixel indexing begins from the top-left and goes rightwards and downwards, accounting for 
all image-pixels which are not masked.

In [ ]:

aplt.plot_array(
    array=lens_subtracted_image, title="Image", positions=mapper.image_plane_data_grid
)
aplt.plot_grid(grid=mapper.source_plane_mesh_grid, title="Source-Plane Mesh Grid")

The reverse mappings of source-pixels to image-pixels can also be used.

If we choose the right source-pixel index, we see multiple imaging: image-pixels in different regions of the
image-plane are grouped into the same source-pixel.

`mappings_from` returns this as a `Mapping`, whose `image_contours` are the boundaries of the connected
image-plane regions that source pixel maps to. Passing them to `regions=` draws them as filled polygons, one
colour per region, so the multiple images of a single source pixel can be read straight off the figure. The
guide `guides/mappings.py` covers this API in full.

We pick the source pixel closest to the centre of the source-plane, which is where the lensed source's light
is reconstructed, because an outskirt pixel maps only to a stray corner of the image and shows nothing. For a
source pixel at the very centre the images merge into one ring-shaped region; a pixel further out maps to
separated arcs.

In [ ]:
mesh_grid = np.asarray(mapper.source_plane_mesh_grid.array)

pix_index = int(np.argmin(np.sum(mesh_grid**2.0, axis=1)))

mappings = mapper.mappings_from(pix_indexes=[[pix_index]])

aaplt.plot_array(
    array=lens_subtracted_image,
    regions=[mapping.image_contours for mapping in mappings],
    title="Image Pixels Which Map To One Source Pixel",
)

__Interpolation__

The regions drawn above cover more image pixels than those whose traced coordinates land inside that source
pixel's own rectangular cell. Pixels further out than the cell are also mapped to this source pixel. 

This is because the mesh uses an interpolation mapping scheme whereby each image pixels is paired with four source 
pixels. For a rectangular mesh, this scheme is called bilinear interpolation, and it means that every pixel maps
not only to the rectangular source pixel it lands in, but also the three neighbouring source pixels. Interpolation is 
key to ensuring that the pixelization can reconstruct smooth source morphologies.

We can confirm that every image pixel maps to four source pixels by printing 
the `pix_sizes_for_sub_slim_index`, which gives the number of mapped source pixels for every image pixel.

We can also confirm that the interpolation introduces weights to each mapping by printing the 
`pix_weights_for_sub_slim_index`, which gives the weight of each mapping for every image pixel.

In [ ]:
print(mapper.pix_sizes_for_sub_slim_index[0:9])
print(mapper.pix_weights_for_sub_slim_index[0:9])

__Over Sampling__

Lets quickly think about what happens when we use over sampling in the pixelization (e.g. `sub_size>1`). For
the `sub_size=1` case above, each image pixel maps to 4 source pixels (due to bilinear interpolation)
with a weight determined from the bilinear interpolation scheme.

However, the default over sampling for a pixelization is `sub_size=4`, meaning each image pixel is divided
into a 4x4 grid of sub-pixels (16 sub-pixels in total). Each of these sub-pixels maps to 4 source pixels
(due to bilinear interpolation), where the weight of each mapping is determined by the bilinear interpolation
scheme divided by 16 (because there are 16 sub-pixels).

This example therefore used a `sub_size=1` to keep the explanation of the likelihood, visualization of the
arrays above and understanding of the mapping scheme as simple as possible. You can manually increase the
`sub_size` above and re-run the notebook to see how this changes the mapping scheme.

__Alternative Meshes__

We can briefly consider how this step differs for other mesh types. Above, we simply overlaid a uniform rectangular
grid to define the source pixel centres and then mapped image pixels to these source pixels.

The `RectangularBilinearAdaptDensity` mesh pretty much works exactly the same, its just that a calculation (which we don't
describe here) works out how to make a grid of rectangular pixels that adapt to the source-plane density and thus
vary in size. 

There is also a `RectangularBilinearAdaptImage` mesh which uses the image of the lensed source to adapt
the rectangular pixel sizes. This often puts even smaller pixels in the brightest regions of the source,
even if it lies offset or away from the caustic.

The adaptive rectangular meshes come in two variants which differ only in the transform used to warp the
uniform grid: the default `RectangularBilinear` meshes use the empirical rank CDF of the traced points (a sort
and a cumulative sum — no extra parameters, fastest on CPUs), whereas the advanced `RectangularRTU` meshes use
a smooth kernel-density CDF — the ray-guided transformed uniform (RTU) grid formulation of Enzi et al. (2026),
https://arxiv.org/abs/2606.30620, which should be cited when using those meshes (the paper pairs the RTU grid
with a Gaussian-process source prior, whereas these examples use PyAutoLens's own regularization schemes such
as `reg.Constant` / `reg.Adapt`). The RTU meshes are recommended on GPUs and are required for gradient-based
(JAX) samplers at the default `over_sample_size_pixelization=1`, where the Bilinear likelihood is exactly
piecewise-constant (zero gradients) — gradient users set `over_sample_size_pixelization >= 4` or use RTU.

There is also a `Delaunay` mesh which uses a Delaunay triangulation to define an irregular grid of source pixels.
This is described fully in the `delaunay` example including a likelihood function guide.

__Mapping Matrix__

The `mapping_matrix` represents the image-pixel to source-pixel mappings above in a 2D matrix. 

It has dimensions `(total_image_pixels, total_source_pixels)`.

In [ ]:
mapping_matrix = al.util.mapper.mapping_matrix_from(
    pix_indexes_for_sub_slim_index=pix_indexes_for_sub_slim_index,
    pix_size_for_sub_slim_index=mapper.pix_sizes_for_sub_slim_index,
    pix_weights_for_sub_slim_index=mapper.pix_weights_for_sub_slim_index,
    pixels=mapper.pixels,
    total_mask_pixels=mapper.source_plane_data_grid.mask.pixels_in_mask,
    slim_index_for_sub_slim_index=mapper.slim_index_for_sub_slim_index,
    sub_fraction=mapper.over_sampler.sub_fraction,
)

A 2D plot of the `mapping_matrix` shows of all image-source pixel mappings.

No row of pixels has more than one non-zero entry. It is not possible for two image pixels to map to the same source 
pixel (meaning that there are no correlated pixels in the mapping matrix).

In [ ]:
plt.imshow(mapping_matrix, aspect=(mapping_matrix.shape[1] / mapping_matrix.shape[0]))
plt.show()
plt.close()

Each column of the `mapping_matrix` can therefore be used to show all image-pixels it maps too. 

For example, above, we plotted all image-pixels of source-pixel 200 (as well as 202 and 204). We can extract all
image-pixel indexes of source pixels 200 using the `mapping_matrix` and use them to plot the image of this
source-pixel (which corresponds to only values of zeros or ones).

In [ ]:
indexes_source_pix_200 = np.nonzero(mapping_matrix[:, 200])

print(indexes_source_pix_200[0])

array_2d = al.Array2D(values=mapping_matrix[:, 200], mask=masked_dataset.mask)

aplt.plot_array(array=array_2d, title="")

__Blurred Mapping Matrix ($f$)__

Each source-pixel can therefore be thought of as an image (where all entries of this image are zeros and ones). 

To incorporate the imaging data's PSF, we simply blur each one of these source-pixel images with the imaging data's 
Point Spread Function (PSF) via 2D convolution.

This operation does not change the dimensions of the mapping matrix, meaning the `blurred_mapping_matrix` also has
dimensions `(total_image_pixels, total_source_pixels)`. It turns the values of zeros and ones into 
non-integer values which have been blurred by the PSF.

In [ ]:
blurred_mapping_matrix = masked_dataset.psf.convolved_mapping_matrix_from(
    mapping_matrix=mapping_matrix, mask=masked_dataset.mask
)

A 2D plot of the `blurred_mapping_matrix` shows all image-source pixel mappings including PSF blurring.

Note how, unlike for the `mapping_matrix`, every row of image-pixels now has multiple non-zero entries. It is now 
possible for two image pixels to map to the same source pixel, because they become correlated by PSF convolution.

In [ ]:
plt.imshow(
    blurred_mapping_matrix,
    aspect=(blurred_mapping_matrix.shape[1] / blurred_mapping_matrix.shape[0]),
)
plt.colorbar()
plt.show()
plt.close()

Each column of the `blurred_mapping_matrix` shows all image-pixels it maps to after PSF blurring. 

In [ ]:
indexes_source_pix_200 = np.nonzero(blurred_mapping_matrix[:, 200])

print(indexes_source_pix_200[0])

array_2d = al.Array2D(values=blurred_mapping_matrix[:, 200], mask=masked_dataset.mask)

aplt.plot_array(array=array_2d, title="")

In Warren & Dye 2003 (https://arxiv.org/abs/astro-ph/0302587) the `blurred_mapping_matrix` is denoted $f_{ij}$
where $i$ maps over all $I$ source pixels and $j$ maps over all $J$ image pixels. 

For example: 

 - $f_{0, 2} = 0.3$ indicates that image-pixel $2$ maps to source-pixel $0$ with a weight of $0.3$ after PSF convolution.
 - $f_{4, 8} = 0$ indicates that image-pixel $8$ does not map to source-pixel $4$, even after PSF convolution.

The indexing of the `mapping_matrix` is reversed compared to the notation of WD03 (e.g. image pixels
are the first entry of `mapping_matrix` whereas for $f$ they are the second index).

In [ ]:
print(f"Mapping between image pixel 0 and source pixel 2 = {mapping_matrix[0, 2]}")

__Data Vector (D)__

To solve for the source pixel fluxes we now pose the problem as a linear inversion.

This requires us to convert the `blurred_mapping_matrix` and our `data` and `noise map` into matrices of certain dimensions. 

The `data_vector`, $D$, is the first matrix and it has dimensions `(total_source_pixels,)`.

In WD03 (https://arxiv.org/abs/astro-ph/0302587) and N15 (https://arxiv.org/abs/1412.7436) the data vector 
is give by: 

 $\vec{D}_{i} = \sum_{\rm  j=1}^{J}f_{ij}(d_{j} - b_{j})/\sigma_{j}^2 \, \, .$

Where:

 - $d_{\rm j}$ are the image-pixel data flux values.
 - $b_{\rm j}$ are the brightness values of the lens light model (therefore $d_{\rm  j} - b_{\rm j}$ is the lens light
 subtracted image).
 - $\sigma{\rm _j}^2$ are the statistical uncertainties of each image-pixel value.

$i$ maps over all $I$ source pixels and $j$ maps over all $J$ image pixels. 

NOTE: WD03 assume the data is already lens subtracted thus $b_{j}$ is omitted (e.g. all values are zero).

In [ ]:
data_vector = al.util.inversion_imaging.data_vector_via_blurred_mapping_matrix_from(
    blurred_mapping_matrix=blurred_mapping_matrix,
    image=np.array(lens_subtracted_image),
    noise_map=np.array(masked_dataset.noise_map),
)

$D$ describes which deconvolved source pixels trace to which image-plane pixels. This ensures the source reconstruction
fully accounts for the PSF when fitting the data.

We can plot $D$ as a column vector:

In [ ]:
plt.imshow(
    data_vector.reshape(data_vector.shape[0], 1), aspect=10.0 / data_vector.shape[0]
)
plt.colorbar()
plt.show()
plt.close()

__Curvature Matrix (F)__

The `curvature_matrix` $F$ is the second matrix and it has dimensions `(total_source_pixels, total_source_pixels)`.

In WD03 / N15 (https://arxiv.org/abs/astro-ph/0302587) the curvature matrix is a 2D matrix given by:

 ${F}_{ik} = \sum_{\rm  j=1}^{J}f_{ij}f_{kj}/\sigma_{j}^2 \, \, .$

NOTE: this notation implicitly assumes a summation over $K$, where $k$ runs over all source-pixel indexes $K$.

Note how summation over $J$ runs over $f$ twice, such that every entry of $F$ is the sum of the multiplication
between all values in every two columns of $f$.

For example, $F_{0,1}$ is the sum of every blurred image pixels values in $f$ of source pixel 0 multiplied by
every blurred image pixel value of source pixel 1.

In [ ]:
curvature_matrix = al.util.inversion.curvature_matrix_via_mapping_matrix_from(
    mapping_matrix=blurred_mapping_matrix, noise_map=masked_dataset.noise_map
)

plt.imshow(curvature_matrix)
plt.colorbar()
plt.show()
plt.close()

For $F_{ik}$ to be non-zero, this requires that the images of source pixels $i$ and $k$ share at least one
image-pixel, which we saw above is only possible due to PSF blurring.

For example, we can see a non-zero entry for $F_{100,101}$ and plotting their images
show overlap.

In [ ]:
source_pixel_0 = 0
source_pixel_1 = 1

print(curvature_matrix[source_pixel_0, source_pixel_1])

array_2d = al.Array2D(
    values=blurred_mapping_matrix[:, source_pixel_0], mask=masked_dataset.mask
)

aplt.plot_array(array=array_2d, title="")

array_2d = al.Array2D(
    values=blurred_mapping_matrix[:, source_pixel_1], mask=masked_dataset.mask
)

aplt.plot_array(array=array_2d, title="")

The following chi-squared is minimized when we perform the inversion and reconstruct the source:

$\chi^2 = \sum_{\rm  j=1}^{J} \bigg[ \frac{(\sum_{\rm  i=1}^{I} s_{i} f_{ij}) + b_{j} - d_{j}}{\sigma_{j}} \bigg]$

Where $s$ is the reconstructed source pixel fluxes in all $I$ source pixels.

The solution for $s$ is therefore given by (equation 5 WD03):

 $s = F^{-1} D$
 
We can compute this using NumPy linear algebra:

In [ ]:

# Because we are no using regularizartion (see below) it is common for the curvature matrix to be singular and lead
# to a LinAlgException. The loop below mitigates this -- you can ignore it as it is not important for understanding
# the PyAutoLens likelihood function.

for i in range(curvature_matrix.shape[0]):
    curvature_matrix[i, i] += 1e-8

reconstruction = np.linalg.solve(curvature_matrix, data_vector)

We can plot this source reconstruction -- it looks like a mess.

The source pixels have noisy and unsmooth values, and it is hard to make out if a source is even being reconstructed. 

In fact, the linear inversion is (over-)fitting noise in the image data, meaning this system of equations is 
ill-posed. We need to apply some form of smoothing on the source reconstruction to avoid over fitting noise.

In [ ]:

aplt.plot_grid(grid=mapper.source_plane_mesh_grid, title="Source-Plane Mesh Grid")

__Regularization Matrix (H)__

Regularization adds a linear regularization term $G_{\rm L}$ to the $\chi^2$ we solve for giving us a new merit 
function $G$ (equation 11 WD03):

 $G = \chi^2 + \lambda \, G_{\rm L}$
 
where $\lambda$ is the `regularization_coefficient` which describes the magnitude of smoothness that is applied. A 
higher $\lambda$ will regularize the source more, leading to a smoother source reconstruction.
 
Different forms for $G_{\rm L}$ can be defined which regularize the source reconstruction in different ways. The 
`Constant` regularization scheme used in this example applies gradient regularization (equation 14 WD03):

 $G_{\rm L} = \sum_{\rm  i}^{I} \sum_{\rm  n=1}^{N}  [s_{i} - s_{i, v}]$

This regularization scheme is easier to express in words -- the summation goes to each source pixel,
determines all source pixels with which it shares a direct vertex (e.g. its neighbors) and penalizes solutions 
where the difference in reconstructed flux of these two neighboring source pixels is large.

The summation does this for all pixels, thus it favours solutions where neighboring source
pixels reconstruct similar values to one another (e.g. it favours a smooth source reconstruction).

We now define the `regularization matrix`, $H$, which allows us to include this smoothing when we solve for $s$. $H$
has dimensions `(total_source_pixels, total_source_pixels)`.

This relates to $G_{\rm L}$ as (equation 13 WD03):

 $H_{ik} = \frac{1}{2} \frac{\partial G_{\rm L}}{\partial s_{i} \partial s_{k}}$

$H$ has the `regularization_coefficient` $\lambda$ folded into it such $\lambda$'s control on the degree of smoothing
is accounted for.

In [ ]:
regularization_matrix = al.util.regularization.constant_regularization_matrix_from(
    coefficient=source_galaxy.pixelization.regularization.coefficient,
    neighbors=mapper.neighbors,
    neighbors_sizes=mapper.neighbors.sizes,
)

We can plot the regularization matrix and note that:

 - non-zero entries indicate that two source-pixels are neighbors and therefore are regularized with one another.
 
 - Zeros indicate the two source pixels do not neighbor one another.
 
The majority of entries are zero, because the majority of source pixels are not neighbors with one another.

In [ ]:
plt.imshow(regularization_matrix)
plt.colorbar()
plt.show()
plt.close()

__F + Lamdba H__

$H$ enters the linear algebra system we solve for as follows (WD03 equation (12)):

 $s = [F + H]^{-1} D$

In [ ]:
curvature_reg_matrix = np.add(curvature_matrix, regularization_matrix)

__Source Reconstruction (s)__

We can now solve the linear system above using NumPy linear algebra. 

Note that the for loop used above to prevent a LinAlgException is no longer required.

In [ ]:
reconstruction = np.linalg.solve(curvature_reg_matrix, data_vector)

By plotting this source reconstruction we can see that regularization has lead us to reconstruct a smoother source,
which actually looks like a galaxy! This also implies we are not over-fitting the noise.

In [ ]:

aplt.plot_grid(grid=mapper.source_plane_mesh_grid, title="Source-Plane Mesh Grid")

__Image Reconstruction__

Using the reconstructed source pixel fluxes we can map the source reconstruction back to the image plane (via
the `blurred mapping_matrix`) and produce a reconstruction of the image data.

In [ ]:
mapped_reconstructed_operated_data = (
    al.util.inversion.mapped_reconstructed_data_via_mapping_matrix_from(
        mapping_matrix=blurred_mapping_matrix, reconstruction=reconstruction
    )
)

mapped_reconstructed_operated_data = al.Array2D(
    values=mapped_reconstructed_operated_data, mask=mask
)

aplt.plot_array(array=mapped_reconstructed_operated_data, title="")

__Likelihood Function__

We now quantify the goodness-of-fit of our lens model and source reconstruction. 

We compute the `log_likelihood` of the fit, which is the value returned by the `log_likelihood_function`.

The likelihood function for lens modeling consists of five terms:

 $-2 \mathrm{ln} \, \epsilon = \chi^2 + s^{T} H s + \mathrm{ln} \, \left[ \mathrm{det} (F + H) \right] - { \mathrm{ln}} \, \left[ \mathrm{det} (H) \right] + \sum_{\rm  j=1}^{J} { \mathrm{ln}} \left [2 \pi (\sigma_j)^2 \right]  \, .$

This expression was first derived by Suyu 2006 (https://arxiv.org/abs/astro-ph/0601493) and is given by equation (19).
It was derived into **PyAutoLens** notation in Dye 2008 (https://arxiv.org/abs/0804.4002) equation (5).

We now explain what each of these terms mean.

__Chi Squared__

The first term is a $\chi^2$ statistic, which is defined above in our merit function as and is computed as follows:

 - `model_data` = `mapped_reconstructed_operated_data` + `lens_light_convolved_image`
 - `residual_map` = (`data` - `model_data`)
 - `normalized_residual_map` = (`data` - `model_data`) / `noise_map`
 - `chi_squared_map` = (`normalized_residuals`) ** 2.0 = ((`data` - `model_data`)**2.0)/(`variances`)
 - `chi_squared` = sum(`chi_squared_map`)

The chi-squared therefore quantifies if our fit to the data is accurate or not. 

High values of chi-squared indicate that there are many image pixels our model did not produce a good fit to the image 
for, corresponding to a fit with a lower likelihood.

In [ ]:
model_image = convolved_image_2d + mapped_reconstructed_operated_data

residual_map = masked_dataset.data - model_image
normalized_residual_map = residual_map / masked_dataset.noise_map
chi_squared_map = normalized_residual_map**2.0

chi_squared = np.sum(chi_squared_map)

print(chi_squared)

The `chi_squared_map` indicates which regions of the image we did and did not fit accurately.

In [ ]:
chi_squared_map = al.Array2D(values=chi_squared_map, mask=mask)

aplt.plot_array(array=chi_squared_map, title="")


__Regularization Term__

The second term, $s^{T} H s$, corresponds to the $\lambda $G_{\rm L}$ regularization term we added to our merit 
function above.

This is the term which sums up the difference in flux of all reconstructed source pixels, and reduces the likelihood of 
solutions where there are large differences in flux (e.g. the source is less smooth and more likely to be 
overfitting noise).

We compute it below via matrix multiplication, noting that the `regularization_coefficient`, $\lambda$, is built into 
the `regularization_matrix` already.

In [ ]:
regularization_term = np.matmul(
    reconstruction.T, np.matmul(regularization_matrix, reconstruction)
)

print(regularization_term)

__Complexity Terms__

Up to this point, it is unclear why we chose a value of `regularization_coefficient=1.0`. 

We cannot rely on the `chi_squared` and `regularization_term` above to optimally choose its value, because increasing 
the `regularization_coefficient` smooths the solution more and therefore:
 
 - Decreases `chi_squared` by fitting the data worse, producing a lower `log_likelihood`.
 
 - Increases the `regularization_term` by penalizing the differences between source pixel fluxes more, again reducing
 the inferred `log_likelihood`.

If we set the regularization coefficient based purely on these two terms, we would set a value of 0.0 and be back where
we started over-fitting noise!

The terms $\left[ \mathrm{det} (F + H) \right]$ and $ - { \mathrm{ln}} \, \left[ \mathrm{det} (H) \right]$ address 
this problem. 

They quantify how complex the source reconstruction is, and penalize solutions where *it is more complex*. Reducing 
the `regularization_coefficient` makes the source reconstruction more complex (because a source that is 
smoothed less uses more flexibility to fit the data better).

These two terms therefore counteract the `chi_squared` and `regularization_term`, so as to attribute a higher
`log_likelihood` to solutions which fit the data with a more smoothed and less complex source (e.g. one with a higher 
`regularization_coefficient`).

In **HowToLens** -> `chapter 3` -> `tutorial_4_bayesian_regularization` we expand on this further and give a more
detailed description of how these different terms impact the `log_likelihood_function`. 

In [ ]:
log_curvature_reg_matrix_term = np.linalg.slogdet(curvature_reg_matrix)[1]
log_regularization_matrix_term = np.linalg.slogdet(regularization_matrix)[1]

print(log_curvature_reg_matrix_term)
print(log_regularization_matrix_term)

__Noise Normalization Term__

Our likelihood function assumes the imaging data consists of independent Gaussian noise in every image pixel.

The final term ins the likelihood function is therefore a `noise_normalization` term, which consists of the sum
of the log of every noise-map value squared. 

Given the `noise_map` is fixed, this term does not change during the lens modeling process and has no impact on the 
model we infer.

In [ ]:
noise_normalization = float(np.sum(np.log(2 * np.pi * masked_dataset.noise_map**2.0)))

__Calculate The Log Likelihood__

We can now, finally, compute the `log_likelihood` of the lens model, by combining the five terms computed above using
the likelihood function defined above.

In [ ]:
log_evidence = float(
    -0.5
    * (
        chi_squared
        + regularization_term
        + log_curvature_reg_matrix_term
        - log_regularization_matrix_term
        + noise_normalization
    )
)

print(log_evidence)

__Fit__

This process to perform a likelihood function evaluation is what is performed in the `FitImaging` object.

In [ ]:
fit = al.FitImaging(
    dataset=masked_dataset,
    tracer=tracer,
    settings=al.Settings(use_border_relocator=True),
)
fit_log_evidence = fit.log_evidence
print(fit_log_evidence)

aplt.subplot_fit_imaging(fit=fit)


__Lens Modeling__

To fit a lens model to data, the likelihood function illustrated in this tutorial is sampled using a
non-linear search algorithm.

The default sampler is the nested sampling algorithm `Nautilus` (https://github.com/johannesulf/nautilus)
multiple MCMC and optimization algorithms are supported.

__Wrap Up__

We have presented a visual step-by-step guide to the **PyAutoLens** likelihood function, which uses a pixelization, 
regularization scheme and inversion to reconstruct the source galaxy.

There are a number of other inputs features which slightly change the behaviour of this likelihood function, which
are described above but worth refreshing ourselves on:

 - **Sub-gridding**: Oversampling the image grid into a finer grid of sub-pixels, which are all individually 
 ray-traced to the source-plane and paired fractionally with each source pixel.
 
 - **Source-plane Interpolation**: Using a RectangularUniform mesh with bilinear interpolation
 to pair each image (sub-)pixel to multiple source-plane pixels with interpolation weights.
 
 - **Source Morphology Pixelization Adaption**: Adapting the pixelization such that is congregates source pixels around
 the source's brightest regions, as opposed to the magnification-based pixelization used here.
 
 - **Luminosity Weighted Regularization**: Using an adaptive regularization coefficient which adapts the level of 
 regularization applied to the source based on its luminosity.